# Fixed-fold LR multistart (Kaggle)

CPU notebook for the current AM/AFM/FM log-coordinate experiment. It pins the code revision, uses a shallow sparse checkout, runs a small smoke pass, then launches 32 fold candidates per model. GPU is not needed. Enable Internet in Kaggle settings. Results go to /kaggle/working/pwl_runs; package and download them before the session ends.

The search perturbs existing fold candidates locally; it is not a global random search. Fold quota does not guarantee successful fits or holdout cycles.

In [ ]:
import os, sys, subprocess, pathlib, json, time
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["OMP_NUM_THREADS"]="1"
os.environ["MKL_NUM_THREADS"]="1"
AVAILABLE_CPUS=len(os.sched_getaffinity(0)) if hasattr(os,"sched_getaffinity") else (os.cpu_count() or 1)
print("Visible CPU cores:",AVAILABLE_CPUS)

In [ ]:
REPO_URL="https://github.com/KirylMP/PWL-NEST.git"
REPO_BRANCH="experiment/lr-transformed-fixedfold-multistart-20261116"
REPO_COMMIT="97a2c40181ad70c43f3c051b993e0da18cd8fa8a"
REPO_DIR=pathlib.Path("/kaggle/working/PWL-NEST")
if not (REPO_DIR/".git").exists():
    subprocess.run(["git","clone","--depth","1","--filter=blob:none","--no-checkout","--branch",REPO_BRANCH,"--single-branch",REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(["git","-C",str(REPO_DIR),"fetch","--depth","1","origin",REPO_COMMIT],check=True)
subprocess.run(["git","-C",str(REPO_DIR),"sparse-checkout","set","scripts","results/coordinate_experiment","results/fold_anchors","paper/data/random_selected"],check=True)
subprocess.run(["git","-C",str(REPO_DIR),"checkout","--detach",REPO_COMMIT],check=True)
required=["scripts/run_transformed_server_batch.py","scripts/summarize_transformed_server_batch.py","scripts/optimize_fixed_fold_cycles.py","scripts/verify_transformed_fold.py","scripts/polish_transformed_fold.py"]
missing=[x for x in required if not (REPO_DIR/x).exists()]
if missing: raise FileNotFoundError(missing)
print("Pinned revision:",subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip())

In [ ]:
try:
    import numpy as np
    import scipy
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","numpy","scipy"],check=True)
    import numpy as np
    import scipy
print("NumPy",np.__version__,"SciPy",scipy.__version__)

## Smoke run

Run once before the long batch. It checks all three log modes with one candidate and one fit evaluation; this only tests the wiring.

In [ ]:
RUNS_DIR=pathlib.Path("/kaggle/working/pwl_runs")
RUNS_DIR.mkdir(parents=True,exist_ok=True)
SMOKE_DIR=RUNS_DIR/"transformed_log_smoke"
cmd=[sys.executable,"-u","scripts/run_transformed_server_batch.py","--output",str(SMOKE_DIR),
     "--combos","AM:log","AFM:log","FM:log","--valid-per-combo","1",
     "--workers",str(min(3,AVAILABLE_CPUS)),"--fold-nfev","100","--fit-nfev","1",
     "--max-attempts","8","--timeout","300"]
if (SMOKE_DIR/"batch.json").exists(): cmd.append("--resume")
subprocess.run(cmd,cwd=REPO_DIR,check=True)
subprocess.run([sys.executable,"scripts/summarize_transformed_server_batch.py",str(SMOKE_DIR)],cwd=REPO_DIR,check=True)

## Configure and launch

Default: 32 distinct folds per model, up to 80 fit evaluations each. Workers default to at most four to limit memory and CPU pressure; adjust if appropriate for the Kaggle session. Re-running the launch cell resumes the same manifest. Do not change numeric settings on resume.

In [ ]:
VALID_PER_MODEL=32
WORKERS=min(4,AVAILABLE_CPUS)
FOLD_NFEV=1200
FIT_NFEV=80
MAX_ATTEMPTS=5000
TIMEOUT=3600
BATCH_NAME="transformed_log_fold32"
BATCH_DIR=RUNS_DIR/BATCH_NAME
LOG_DIR=RUNS_DIR/"logs"
LOG_DIR.mkdir(parents=True,exist_ok=True)
BATCH_LOG=LOG_DIR/f"{BATCH_NAME}.log"
PID_FILE=LOG_DIR/f"{BATCH_NAME}.pid"
print("Models AM/AFM/FM; total fold quota:",VALID_PER_MODEL*3,"workers:",WORKERS)

In [ ]:
cmd=[sys.executable,"-u","scripts/run_transformed_server_batch.py","--output",str(BATCH_DIR),
     "--combos","AM:log","AFM:log","FM:log",
     "--valid-per-combo",str(VALID_PER_MODEL),"--workers",str(WORKERS),
     "--fold-nfev",str(FOLD_NFEV),"--fit-nfev",str(FIT_NFEV),
     "--max-attempts",str(MAX_ATTEMPTS),"--timeout",str(TIMEOUT)]
if (BATCH_DIR/"batch.json").exists(): cmd.append("--resume")
if PID_FILE.exists():
    old=PID_FILE.read_text().strip()
    active=subprocess.run(["ps","-p",old,"-o","args="],text=True,capture_output=True).stdout
    if "run_transformed_server_batch.py" in active:
        raise RuntimeError("Batch appears active; monitor it instead of launching a duplicate.")
with BATCH_LOG.open("a",buffering=1) as log:
    proc=subprocess.Popen(cmd,cwd=REPO_DIR,stdout=log,stderr=subprocess.STDOUT,start_new_session=True,close_fds=True)
PID_FILE.write_text(str(proc.pid)+"\n")
print("Started PID",proc.pid,"log",BATCH_LOG)

## Monitor

Run this cell repeatedly. It prints recent log lines and the current per-model summary.

In [ ]:
subprocess.run(["tail","-n","40",str(BATCH_LOG)],check=False)
subprocess.run([sys.executable,"scripts/summarize_transformed_server_batch.py",str(BATCH_DIR)],cwd=REPO_DIR,check=False)
if PID_FILE.exists():
    subprocess.run(["ps","-p",PID_FILE.read_text().strip(),"-o","pid,etime,time,%cpu,stat,args"],check=False)

## Package results

Run after the batch finishes. The ZIP contains search statuses, fitted candidates, reports, verification files, summaries and logs.

In [ ]:
import shutil
from IPython.display import FileLink, display
RESULT_ZIP=pathlib.Path("/kaggle/working")/f"{BATCH_NAME}.zip"
if RESULT_ZIP.exists(): RESULT_ZIP.unlink()
shutil.make_archive(str(RESULT_ZIP.with_suffix("")),"zip",root_dir=BATCH_DIR)
print("ZIP size MiB:",round(RESULT_ZIP.stat().st_size/(1024**2),2))
display(FileLink(str(RESULT_ZIP)))